# 04 | Fake News Detection mit lokalen LLMs (Ollama)

Ergänzung zu Notebook **01_fake_news_supervised.ipynb** (klassisches Supervised Learning mit TF-IDF + Klassifikatoren).

**Ziel:** Klassifikation von Nachrichtenartikeln als *FAKE* oder *TRUE* mit verschiedenen kleinen, lokal lauffähigen Sprachmodellen via [Ollama](https://ollama.com) – plus eine semantische Analyse über Embeddings.

## Inhalt
1. Setup & Verbindung zum lokalen Ollama-Server
2. Modellauswahl (kleine Modelle: 1B–3B Parameter)
3. Daten laden & stratifizierte Stichprobe
4. Prompt-Engineering & Klassifikationsfunktion
5. Inferenz pro Modell (Zero-Shot)
6. Auswertung: Accuracy, F1, Confusion Matrix, Modellvergleich
7. Semantische Analyse mit Embeddings (PCA / t-SNE)
8. Fazit

**Voraussetzung:** Ollama lokal installiert und gestartet (`ollama serve`). Die benötigten Modelle werden im Notebook automatisch gepullt.

## 1 | Setup & Verbindung

In [1]:
import sys; sys.path.insert(0, '..')
import json
import time
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import ollama

from src.data_loading import load_fake_news_data, set_seeds
from src.plotting import setup_plot_style

cfg = yaml.safe_load(open('../configs/default.yaml'))
set_seeds(cfg['random_seed'])
setup_plot_style(cfg)

/opt/homebrew/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Verbindung zum lokalen Ollama-Server pruefen
try:
    available = [m['model'] for m in ollama.list()['models']]
    print(f'Ollama erreichbar. Bereits installierte Modelle ({len(available)}):')
    for m in available:
        print(f'  - {m}')
except Exception as e:
    print('Ollama nicht erreichbar. Bitte `ollama serve` starten.')
    print(e)

Ollama erreichbar. Bereits installierte Modelle (9):
  - nomic-embed-text:latest
  - gemma4:e4b
  - gemma3:12b
  - x/z-image-turbo:latest
  - llama3.2:3b
  - dimavz/whisper-tiny:latest
  - legraphista/Orpheus:latest
  - gpt-oss:20b
  - mistral:latest


## 2 | Modellauswahl

Wir vergleichen mehrere kleine Open-Weight-Modelle. Alle laufen auf einem normalen Laptop (8–16 GB RAM).

In [3]:
MODELS = [
    'llama3.2:1b',
    'llama3.2:3b',
    'qwen2.5:3b',
    'phi3:mini',
    'gemma2:2b',
]
EMBEDDING_MODEL = 'nomic-embed-text'

# Falls ein Modell noch nicht lokal vorhanden ist: pullen.
def ensure_model(name):
    installed = {m['model'] for m in ollama.list()['models']}
    if name in installed or any(m.startswith(name) for m in installed):
        return
    print(f'Pull {name} ...')
    ollama.pull(name)

for m in MODELS + [EMBEDDING_MODEL]:
    ensure_model(m)
print('Alle Modelle bereit.')

Pull llama3.2:1b ...
Pull qwen2.5:3b ...
Pull phi3:mini ...
Pull gemma2:2b ...
Alle Modelle bereit.


## 3 | Daten laden & stratifizierte Stichprobe

Die LLM-Inferenz ist deutlich teurer als TF-IDF + Logistische Regression. Wir ziehen eine stratifizierte Stichprobe von ~200 Artikeln (jeweils zur Hälfte fake / true). Wer mehr Rechenleistung hat, kann `SAMPLE_SIZE` erhöhen.

In [4]:
SAMPLE_SIZE = 200          # Gesamtanzahl Artikel
MAX_CHARS = 1500           # Text-Truncation, um Prompt-Laenge zu begrenzen

df = load_fake_news_data(cfg)
print(f'Gesamt-Datensatz: {len(df):,} Artikel')
print(df.label.value_counts().rename({0: 'fake', 1: 'true'}))

Gesamt-Datensatz: 44,898 Artikel
label
fake    23481
true    21417
Name: count, dtype: int64


In [5]:
# Stratifizierte Stichprobe: gleich viele fake/true
n_per_class = SAMPLE_SIZE // 2
sample = (
    df.groupby('label', group_keys=False)
      .apply(lambda g: g.sample(n=n_per_class, random_state=cfg['random_seed']))
      .reset_index(drop=True)
)
sample['text_short'] = (sample['title'].fillna('') + '\n\n' + sample['text'].fillna('')).str[:MAX_CHARS]
print(f'Stichprobe: {len(sample)} Artikel, {sample.label.value_counts().to_dict()}')
sample.head(3)

Stichprobe: 200 Artikel, {0: 100, 1: 100}


/var/folders/28/rx0zfkhs42j2lb1l_v0x2l740000gn/T/ipykernel_24680/1489979773.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=n_per_class, random_state=cfg['random_seed']))


,title,text,subject,date,label,text_short
0,"NY Daily News Goes ALL IN On NRA, ‘Thanks’ Th...",The NY Daily News is never shy when it comes t...,News,"June 13, 2016",0,"NY Daily News Goes ALL IN On NRA, ‘Thanks’ Th..."
1,IT’S ABOUT TIME! 19 GOP Congressmen Call For C...,All we can say about this development is that ...,politics,"Oct 13, 2017",0,IT’S ABOUT TIME! 19 GOP Congressmen Call For C...
2,INSANE ANTI-TRUMP PROTESTER Lights Trump Suppo...,The Inauguration was a beautiful day full of g...,left-news,"Jan 23, 2017",0,INSANE ANTI-TRUMP PROTESTER Lights Trump Suppo...


## 4 | Prompt-Engineering & Klassifikationsfunktion

Wir verwenden einen **Zero-Shot-Prompt** mit erzwungenem JSON-Format (`format='json'`).
Das Modell soll Label, Konfidenz und kurze Begruendung zurueckgeben.

In [6]:
SYSTEM_PROMPT = (
    'Du bist ein erfahrener Faktenpruefer fuer englischsprachige Nachrichten. '
    'Du analysierst einen Artikel und entscheidest, ob er sehr wahrscheinlich '
    'eine echte Nachricht (TRUE) oder Fake News (FAKE) ist. '
    'Achte auf Sensationsrhetorik, fehlende Quellen, einseitige Darstellung, '
    'reisserische Sprache und unbelegte Behauptungen. '
    'Antworte AUSSCHLIESSLICH als JSON: '
    '{"label": "FAKE" oder "TRUE", "confidence": Zahl zwischen 0 und 1, "reason": "kurze Begruendung"}.'
)

def classify(model: str, text: str, timeout_s: float = 60.0) -> dict:
    """Klassifiziert einen Artikel mit einem Ollama-Modell. Gibt dict mit label, confidence, reason, latency."""
    t0 = time.time()
    try:
        resp = ollama.chat(
            model=model,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': f'Artikel:\n\n{text}'},
            ],
            format='json',
            options={'temperature': 0.0, 'num_predict': 200},
        )
        raw = resp['message']['content']
        parsed = json.loads(raw)
        label = str(parsed.get('label', '')).upper().strip()
        if label not in {'FAKE', 'TRUE'}:
            label = 'PARSE_ERROR'
        return {
            'label_pred': label,
            'confidence': float(parsed.get('confidence', 0.0)),
            'reason': str(parsed.get('reason', ''))[:300],
            'latency_s': time.time() - t0,
        }
    except Exception as e:
        return {'label_pred': 'ERROR', 'confidence': 0.0, 'reason': str(e)[:300], 'latency_s': time.time() - t0}

# Smoke-Test mit einem Sample
demo = classify(MODELS[0], sample.iloc[0]['text_short'])
print('Demo-Antwort:', demo)
print('Wahres Label:', 'TRUE' if sample.iloc[0]['label'] == 1 else 'FAKE')

Demo-Antwort: {'label_pred': 'FAKE', 'confidence': 0.0, 'reason': 'Fehlende Quelle, einseitige Darstellung und reisserische Sprache sowie unbelegte Behauptungen.', 'latency_s': 11.213368892669678}
Wahres Label: FAKE


## 5 | Inferenz pro Modell

Wir loopen ueber alle Modelle und alle Samples. Ergebnisse werden inkrementell in `data/processed/llm_predictions.parquet` persistiert, damit ein Abbruch nicht den Fortschritt verliert.

In [ ]:
OUT_PATH = Path('../data/processed/llm_predictions.parquet')
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Bereits berechnete (model, sample_idx)-Paare wiederverwenden
if OUT_PATH.exists():
    results_df = pd.read_parquet(OUT_PATH)
    done = set(zip(results_df['model'], results_df['sample_idx']))
    print(f'Bereits {len(results_df)} Ergebnisse geladen.')
else:
    results_df = pd.DataFrame()
    done = set()

rows = []
for model in MODELS:
    print(f'\nModell: {model}')
    for idx, row in tqdm(sample.iterrows(), total=len(sample)):
        if (model, idx) in done:
            continue
        out = classify(model, row['text_short'])
        rows.append({
            'model': model,
            'sample_idx': idx,
            'label_true': 'TRUE' if row['label'] == 1 else 'FAKE',
            **out,
        })
        # Inkrementell speichern alle 25 Antworten
        if len(rows) % 25 == 0:
            tmp = pd.concat([results_df, pd.DataFrame(rows)], ignore_index=True)
            tmp.to_parquet(OUT_PATH, index=False)

if rows:
    results_df = pd.concat([results_df, pd.DataFrame(rows)], ignore_index=True)
    results_df.to_parquet(OUT_PATH, index=False)
print(f'\nGesamt: {len(results_df)} Vorhersagen gespeichert in {OUT_PATH}')


Modell: llama3.2:1b


100%|██████████| 200/200 [03:27<00:00,  1.04s/it]



Modell: llama3.2:3b


 65%|██████▌   | 130/200 [06:49<03:46,  3.24s/it]

## 6 | Auswertung

### 6.1 Metriken pro Modell

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def evaluate(group: pd.DataFrame) -> pd.Series:
    valid = group[group['label_pred'].isin(['FAKE', 'TRUE'])]
    n_total = len(group)
    n_valid = len(valid)
    if n_valid == 0:
        return pd.Series({'accuracy': np.nan, 'f1_fake': np.nan, 'f1_true': np.nan,
                          'parse_rate': 0.0, 'avg_latency_s': group['latency_s'].mean(), 'n': n_total})
    y_true = valid['label_true']
    y_pred = valid['label_pred']
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=['FAKE', 'TRUE'], zero_division=0)
    return pd.Series({
        'accuracy': acc,
        'f1_fake': f1[0],
        'f1_true': f1[1],
        'parse_rate': n_valid / n_total,
        'avg_latency_s': group['latency_s'].mean(),
        'n': n_total,
    })

metrics = results_df.groupby('model').apply(evaluate).sort_values('accuracy', ascending=False)
metrics

### 6.2 Vergleichsplot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
metrics[['accuracy', 'f1_fake', 'f1_true']].plot.bar(ax=axes[0])
axes[0].set_title('Klassifikationsgüte pro Modell')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
axes[0].axhline(0.5, color='grey', ls='--', alpha=0.5, label='Zufallsbaseline')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=30)

metrics['avg_latency_s'].plot.bar(ax=axes[1], color='steelblue')
axes[1].set_title('Durchschnittliche Latenz pro Sample (Sekunden)')
axes[1].set_ylabel('Sekunden')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### 6.3 Confusion Matrix pro Modell

In [ ]:
n_models = len(MODELS)
cols = 3
rows_fig = (n_models + cols - 1) // cols
fig, axes = plt.subplots(rows_fig, cols, figsize=(4 * cols, 3.5 * rows_fig))
axes = np.array(axes).flatten()

for ax, model in zip(axes, MODELS):
    sub = results_df[(results_df['model'] == model) & results_df['label_pred'].isin(['FAKE', 'TRUE'])]
    if sub.empty:
        ax.set_visible(False)
        continue
    cm = confusion_matrix(sub['label_true'], sub['label_pred'], labels=['FAKE', 'TRUE'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['FAKE', 'TRUE'], yticklabels=['FAKE', 'TRUE'], ax=ax)
    ax.set_title(model)
    ax.set_xlabel('Vorhergesagt')
    ax.set_ylabel('Wahr')

for ax in axes[len(MODELS):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

### 6.4 Beispiele für Fehlklassifikationen

Ein Blick in die Begründungen der Modelle zeigt typische Fehlermuster (Sarkasmus, Satire, kurze Texte ohne Kontext).

In [ ]:
errors = results_df[
    (results_df['label_pred'].isin(['FAKE', 'TRUE'])) &
    (results_df['label_pred'] != results_df['label_true'])
].copy()
errors = errors.merge(sample[['title']], left_on='sample_idx', right_index=True, how='left')
errors[['model', 'title', 'label_true', 'label_pred', 'confidence', 'reason']].head(10)

## 7 | Semantische Analyse mit Embeddings

Zusätzlich zur LLM-Klassifikation untersuchen wir, ob fake und true News bereits im Embedding-Raum trennbar sind. Wir verwenden `nomic-embed-text` und visualisieren die Struktur mit PCA und t-SNE.

In [ ]:
EMB_PATH = Path('../data/processed/llm_embeddings.parquet')

if EMB_PATH.exists():
    emb_df = pd.read_parquet(EMB_PATH)
    print(f'{len(emb_df)} Embeddings aus Cache geladen.')
else:
    embeddings = []
    for idx, row in tqdm(sample.iterrows(), total=len(sample), desc='Embeddings'):
        try:
            r = ollama.embeddings(model=EMBEDDING_MODEL, prompt=row['text_short'])
            embeddings.append({'sample_idx': idx, 'label': row['label'], 'embedding': r['embedding']})
        except Exception as e:
            print(f'Fehler bei Sample {idx}: {e}')
    emb_df = pd.DataFrame(embeddings)
    emb_df.to_parquet(EMB_PATH, index=False)
    print(f'{len(emb_df)} Embeddings gespeichert.')

X_emb = np.vstack(emb_df['embedding'].values)
y_emb = emb_df['label'].values
print('Embedding-Matrix:', X_emb.shape)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

pca = PCA(n_components=2, random_state=cfg['random_seed'])
X_pca = pca.fit_transform(X_emb)

tsne = TSNE(n_components=2, perplexity=30, random_state=cfg['random_seed'], init='pca', learning_rate='auto')
X_tsne = tsne.fit_transform(X_emb)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
label_names = np.where(y_emb == 1, 'TRUE', 'FAKE')
for ax, X_red, title in [(axes[0], X_pca, f'PCA (Varianz erklaert: {pca.explained_variance_ratio_.sum():.2%})'),
                          (axes[1], X_tsne, 't-SNE')]:
    sns.scatterplot(x=X_red[:, 0], y=X_red[:, 1], hue=label_names, ax=ax, alpha=0.7, palette={'FAKE': '#d62728', 'TRUE': '#2ca02c'})
    ax.set_title(title)
    ax.set_xlabel('Komponente 1')
    ax.set_ylabel('Komponente 2')
plt.tight_layout()
plt.show()

### 7.1 Lineare Trennbarkeit der Embeddings

Wenn die beiden Klassen im Embedding-Raum trennbar sind, sollte schon eine einfache Logistische Regression auf den Embeddings hohe Genauigkeit liefern – ein Vergleich zur Zero-Shot-LLM-Klassifikation.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

clf = LogisticRegression(max_iter=1000, random_state=cfg['random_seed'])
scores = cross_val_score(clf, X_emb, y_emb, cv=5, scoring='accuracy')
print(f'Logistische Regression auf Embeddings (5-fold CV): {scores.mean():.3f} ± {scores.std():.3f}')
print('Vergleich – beste LLM-Accuracy:', f"{metrics['accuracy'].max():.3f} ({metrics['accuracy'].idxmax()})")

## 8 | Fazit

**Was wir gesehen haben:**
- Kleine Open-Weight-LLMs (1B–3B) können Fake-News-Klassifikation **zero-shot** ohne Training versuchen – die Genauigkeit variiert stark zwischen Modellen.
- Embedding-basierte Klassifikation (nomic-embed-text + LogReg) ist meist deutlich schneller und oft konkurrenzfähig.
- Die klassische Pipeline aus Notebook 01 (TF-IDF + LogReg) bleibt für diesen Datensatz extrem stark – LLM-Ansätze glänzen eher bei kleinen Datensätzen ohne Trainingsdaten.

**Limitierungen:**
- Stichprobe von 200 Samples ist klein – Konfidenzintervalle der Metriken sind breit.
- Der Datensatz stammt aus einer engen Quellenstruktur (Reuters vs. unzuverlässige Quellen) – die Modelle könnten Stilmerkmale statt Faktizität lernen.
- Zero-shot Prompts sind sensibel gegenüber Formulierung – ein Few-Shot-Ansatz könnte die Werte verbessern.

**Erweiterungsideen:** Few-Shot Prompting, Chain-of-Thought, Modell-Ensemble (Mehrheitsentscheidung), Vergleich mit größeren Modellen über die Anthropic API.